# Sanity checks

Run after `prepare_games.py`. Anything odd here is cheaper to fix now than after training.

In [ ]:
import pandas as pd

df = pd.read_parquet("games/part-000.parquet")  # one shard is plenty for sanity checks
df["n_moves"] = df.moves.map(len)
df["base"] = df.TimeControl.str.split("+").str[0].astype(int)
print(f"{len(df):,} games")
df.describe()

Ratings, and the gap between the two players. Matchmaking should keep the gap narrow.

In [ ]:
df[["WhiteElo", "BlackElo"]].plot.hist(bins=60, alpha=0.5, title="Ratings");

In [ ]:
(df.WhiteElo - df.BlackElo).plot.hist(bins=100, title="White minus Black rating");

Game length, time controls, and the move vocabulary (the number of tokens the model will see).

In [ ]:
df.n_moves.plot.hist(bins=100, title="Half-moves per game");

In [ ]:
print(df.moves.explode().nunique(), "distinct SAN moves")
df.TimeControl.value_counts().head(10)

Clocks: the first clock should equal the base time, none should be negative, and White's clock should mostly fall.

In [ ]:
print("first clock == base time:", (df.clocks.map(lambda c: c[0]) == df.base).mean())
print("negative clocks:", (df.clocks.map(min) < 0).sum())
pd.DataFrame([list(c[0::2]) for c in df.clocks.head(5)]).T.plot(
    title="White's clock, first 5 games", xlabel="White's move", ylabel="seconds");